# **Test Models**

In [ ]:
# === Environment setup: clone repo and install dependencies (Colab/Jupyter) ===
!git clone https://github.com/guimatiolli/EV-soc-battery-ei-lstm-kf.git
%cd EV-soc-battery-ei-lstm-kf

!pip -q install xlsxwriter

import os
print("CWD:", os.getcwd())
print("Repo files:", os.listdir(".")[:20])
print("Has data?:", os.path.exists("data"))
print("Has artefatos?:", os.path.exists("artefatos"))
print("Has modelos?:", os.path.exists("modelos"))


Cloning into 'EV-soc-battery-ei-lstm-kf'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 60 (delta 23), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 14.80 MiB | 17.92 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/EV-soc-battery-ei-lstm-kf
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.1 MB/s eta 0:00:00
CWD: /content/EV-soc-battery-ei-lstm-kf
Repo files: ['.git', 'data', 'modelos', 'README.md', 'LICENSE', 'code', 'artefatos']
Has data?: True
Has artefatos?: True
Has modelos?: True


In [ ]:
import os, time, psutil, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Input, Dropout, Conv1D, Add,
    LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR   = "data"
MODELS_DIR = "modelos"
ART_DIR    = "artefatos"
OUT_DIR    = "relatorios"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

TIME_STEP     = 50
INPUT_COLS    = ['Current(A)', 'Voltage(V)', 'média móvel', 'Temperature']
OUTPUT_COL    = 'SOC'
TEMP_JUMP_THR = 0.5

DATA_TREINO = f"{DATA_DIR}/teste3_LSTM.xlsx"
ABA_TREINO  = "DST"

XSCALER_OUT = f"{ART_DIR}/x_scaler.pkl"
YSCALER_OUT = f"{ART_DIR}/y_scaler.pkl"

TEMPS  = [10, 20, 30, 40]
CICLOS = ["US06", "FUDS"]

print("✅ Setup OK")


✅ Setup OK


In [ ]:
def carregar_df(arquivo, aba):
    df = pd.read_excel(arquivo, sheet_name=aba)
    if 'Temperature (C)_1' in df.columns:
        df = df.rename(columns={'Temperature (C)_1': 'Temperature'})
    return df

def criar_janelas_por_bloco_raw(df, time_step=TIME_STEP, thr=TEMP_JUMP_THR):
    X_list, y_list = [], []
    df_ = df.copy()
    df_['temp_diff'] = df_['Temperature'].diff().abs()
    limiares = df_.index[df_['temp_diff'] > thr].tolist()
    blocos = [0] + limiares + [len(df_)]

    for i in range(len(blocos) - 1):
        ini, fim = blocos[i], blocos[i+1]
        bloco = df_.iloc[ini:fim]
        X_block = bloco[INPUT_COLS].values
        y_block = bloco[[OUTPUT_COL]].values

        for j in range(len(bloco) - time_step):
            X_list.append(X_block[j:j+time_step, :])
            y_list.append(y_block[j+time_step, 0])

    return np.asarray(X_list), np.asarray(y_list)

def criar_janelas_seq(X_raw, y_raw, time_step=TIME_STEP):
    X_seq, y_seq = [], []
    for i in range(len(X_raw) - time_step):
        X_seq.append(X_raw[i:i+time_step])
        y_seq.append(y_raw[i+time_step])
    return np.asarray(X_seq), np.asarray(y_seq)

def metricas(y_ref, y_hat):
    rmse = float(np.sqrt(mean_squared_error(y_ref, y_hat)))
    mae  = float(mean_absolute_error(y_ref, y_hat))
    maxe = float(np.max(np.abs(y_ref - y_hat)))
    r2   = float(r2_score(y_ref, y_hat))
    return rmse, mae, maxe, r2

def medir_tempo(func, *args, **kwargs):
    process = psutil.Process(os.getpid())
    cpu_i = process.cpu_times()
    mem_i = process.memory_info().rss/(1024*1024)
    t0 = time.time()
    out = func(*args, **kwargs)
    t1 = time.time()
    cpu_f = process.cpu_times()
    mem_f = process.memory_info().rss/(1024*1024)
    return out, (t1-t0), (cpu_f.user-cpu_i.user), (mem_f-mem_i)

def fx(x, dt): return x
def hx(x): return x

def kf_identity(y, process_std, meas_std, P0=1.0, dt=1.0, clip=True):
    z = np.asarray(y, float).ravel()
    if z.size == 0:
        return np.array([])
    Q, R = float(process_std**2), float(meas_std**2)
    x, P = float(z[0]), float(P0)
    out = [x]
    for k in range(1, len(z)):
        P = P + Q
        yk = float(z[k]) - x
        S  = P + R
        K  = P / max(S, 1e-12)
        x  = x + K * yk
        P  = (1.0 - K) * P
        if clip:
            x = float(np.clip(x, 0.0, 1.0))
        out.append(x)
    return np.asarray(out, float)

def estimar_Q_R(y_true_inv, y_pred_inv):
    res = y_pred_inv - y_true_inv
    pL, pH = np.percentile(res, [0.5, 99.5])
    res_c = np.clip(res, pL, pH)
    meas_std = float(np.std(res_c, ddof=1)); meas_std = max(meas_std, 1e-6)

    d = np.diff(y_true_inv)
    if len(d) > 0:
        qL, qH = np.percentile(d, [0.5, 99.5])
        d_c = np.clip(d, qL, qH)
        process_std = float(np.std(d_c, ddof=1)); process_std = max(process_std, 1e-7)
    else:
        process_std = 1e-4

    return process_std, meas_std

print("✅ Utilities OK")


✅ Utilities OK


In [ ]:
df_dst = pd.read_excel(DATA_TREINO, sheet_name=ABA_TREINO)

X_seq_raw, y_seq_raw = criar_janelas_por_bloco_raw(df_dst, time_step=TIME_STEP, thr=TEMP_JUMP_THR)
print(f"Total DST windows (raw): {X_seq_raw.shape[0]}")
print(f"X_seq_raw: {X_seq_raw.shape} | y_seq_raw: {y_seq_raw.shape}")

X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_seq_raw, y_seq_raw, test_size=0.2, random_state=SEED, shuffle=True
)
print(f"Train windows: {X_train_raw.shape[0]} | Val windows: {X_val_raw.shape[0]}")

n_tr, T, F = X_train_raw.shape
x_scaler = MinMaxScaler(feature_range=(-1, 1))
y_scaler = MinMaxScaler(feature_range=(0, 1))

X_train_flat = X_train_raw.reshape(-1, F)
x_scaler.fit(X_train_flat)

y_train_flat = y_train_raw.reshape(-1, 1)
y_scaler.fit(y_train_flat)

X_train = x_scaler.transform(X_train_flat).reshape(n_tr, T, F)
y_train = y_scaler.transform(y_train_flat).reshape(-1)

n_val = X_val_raw.shape[0]
X_val = x_scaler.transform(X_val_raw.reshape(-1, F)).reshape(n_val, T, F)
y_val = y_scaler.transform(y_val_raw.reshape(-1, 1)).reshape(-1)

joblib.dump(x_scaler, XSCALER_OUT)
joblib.dump(y_scaler, YSCALER_OUT)

print("✅ Shapes:")
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "| y_val  :", y_val.shape)
print("✅ Scalers saved:", XSCALER_OUT, YSCALER_OUT)


Total DST windows (raw): 27956
X_seq_raw: (27956, 50, 4) | y_seq_raw: (27956,)
Train windows: 22364 | Val windows: 5592
✅ Shapes:
X_train: (22364, 50, 4) | y_train: (22364,)
X_val  : (5592, 50, 4) | y_val  : (5592,)
✅ Scalers saved: artefatos/x_scaler.pkl artefatos/y_scaler.pkl


In [ ]:
def make_callbacks(model_name: str, monitor="val_loss", patience=50, verbose=0):
    model_out_path = f"{MODELS_DIR}/{model_name}_seu.keras"

    early_model = EarlyStopping(
        monitor=monitor,
        patience=patience,
        restore_best_weights=True
    )

    ckpt_model = ModelCheckpoint(
        filepath=model_out_path,
        monitor=monitor,
        save_best_only=True,
        verbose=verbose
    )

    return early_model, ckpt_model, model_out_path

models_trained = {}
histories = {}

print("✅ make_callbacks OK")


✅ make_callbacks OK


LSTM

In [ ]:
early_lstm, ckpt_lstm, MODELO_OUT_LSTM = make_callbacks("LSTM_30x1")

model_lstm = Sequential([
    LSTM(30, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(1, activation='sigmoid')
])
model_lstm.compile(optimizer='adam', loss='mse')

history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=150,
    batch_size=64,
    verbose=1,
    validation_data=(X_val, y_val),
    shuffle=True,
    callbacks=[early_lstm, ckpt_lstm]
)

model_lstm = load_model(MODELO_OUT_LSTM)
models_trained["LSTM"] = model_lstm
histories["LSTM"] = history_lstm
print("✅ Best LSTM loaded:", MODELO_OUT_LSTM)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - loss: 0.0349 - val_loss: 0.0012
Epoch 2/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - loss: 0.0012 - val_loss: 0.0011
Epoch 3/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - loss: 0.0010 - val_loss: 0.0011
Epoch 4/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - loss: 9.6697e-04 - val_loss: 0.0011
Epoch 5/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 9.1204e-04 - val_loss: 0.0011
Epoch 6/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - loss: 8.5505e-04 - val_loss: 0.0011
Epoch 7/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - loss: 7.8586e-04 - val_loss: 0.0010
Epoch 8/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 18s 51ms/step - loss: 6.6353e-04 - val_loss: 4.5336e-04
Epoch 9/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - loss: 4.4333e-04 - val_loss: 2.7308e-04
Epoch 10/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 19s 40ms/step - loss: 3.7337e-04 - val_loss: 2.4902e-04
Epoch 11/150
350/350 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - loss: 3.4470e-0

GRU

In [ ]:
early_gru, ckpt_gru, MODELO_OUT_GRU = make_callbacks("GRU_30x1")

model_gru = Sequential([
    GRU(30, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(1, activation='sigmoid')
])
model_gru.compile(optimizer='adam', loss='mse')

history_gru = model_gru.fit(
    X_train, y_train,
    epochs=150,
    batch_size=64,
    verbose=1,
    validation_data=(X_val, y_val),
    shuffle=True,
    callbacks=[early_gru, ckpt_gru]
)

model_gru = load_model(MODELO_OUT_GRU)
models_trained["GRU"] = model_gru
histories["GRU"] = history_gru
print("✅ Best GRU loaded:", MODELO_OUT_GRU)


TCN

In [ ]:
early_tcn, ckpt_tcn, MODELO_OUT_TCN = make_callbacks("TCN")

def build_tcn(input_shape, num_filters=32, kernel_size=3, num_layers=3, dropout=0.1):
    inputs = Input(shape=input_shape)
    x = inputs

    for i in range(num_layers):
        dilation_rate = 2 ** i
        res = x

        x = Conv1D(num_filters, kernel_size, padding='causal', dilation_rate=dilation_rate, activation='relu')(x)
        x = Dropout(dropout)(x)
        x = Conv1D(num_filters, kernel_size, padding='causal', dilation_rate=dilation_rate, activation='relu')(x)
        x = Dropout(dropout)(x)

        if res.shape[-1] != num_filters:
            res = Conv1D(num_filters, 1, padding='same')(res)

        x = Add()([x, res])
        x = LayerNormalization()(x)

    x = x[:, -1, :]
    outputs = Dense(1, activation='sigmoid')(x)
    return Model(inputs, outputs, name="TCN_SOC")

model_tcn = build_tcn((X_train.shape[1], X_train.shape[2]), num_filters=32, kernel_size=3, num_layers=3, dropout=0.1)
model_tcn.compile(optimizer='adam', loss='mse')

history_tcn = model_tcn.fit(
    X_train, y_train,
    epochs=150,
    batch_size=64,
    verbose=1,
    validation_data=(X_val, y_val),
    shuffle=True,
    callbacks=[early_tcn, ckpt_tcn]
)

model_tcn = load_model(MODELO_OUT_TCN)
models_trained["TCN"] = model_tcn
histories["TCN"] = history_tcn
print("✅ Best TCN loaded:", MODELO_OUT_TCN)


TRANSFORMES

In [ ]:
early_tr, ckpt_tr, MODELO_OUT_TR = make_callbacks("Transformer")

@tf.keras.utils.register_keras_serializable()
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_len = int(max_len)
        self.d_model = int(d_model)
        self.pos_emb = tf.keras.layers.Embedding(self.max_len, self.d_model)

    def call(self, x):
        positions = tf.range(tf.shape(x)[1])
        return x + self.pos_emb(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"max_len": self.max_len, "d_model": self.d_model})
        return cfg

def build_light_transformer(input_shape, d_model=32, num_heads=4, num_layers=2, ff_dim=64, dropout=0.1):
    inputs = Input(shape=input_shape)
    x = Dense(d_model)(inputs)
    x = PositionalEncoding(input_shape[0], d_model)(x)
    x = Dropout(dropout)(x)

    for _ in range(num_layers):
        attn = MultiHeadAttention(num_heads=num_heads, key_dim=max(1, d_model//num_heads))(x, x)
        attn = Dropout(dropout)(attn)
        x = LayerNormalization()(x + attn)

        ff = Dense(ff_dim, activation='gelu')(x)
        ff = Dropout(dropout)(ff)
        ff = Dense(d_model)(ff)
        ff = Dropout(dropout)(ff)
        x = LayerNormalization()(x + ff)

    x = GlobalAveragePooling1D()(x)
    x = Dense(32, activation='gelu')(x)
    x = Dropout(dropout)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    return Model(inputs, outputs, name="LightTransformer_SOC")

model_tr = build_light_transformer((X_train.shape[1], X_train.shape[2]), d_model=32, num_heads=4, num_layers=2, ff_dim=64, dropout=0.1)
model_tr.compile(optimizer='adam', loss='mse')

history_tr = model_tr.fit(
    X_train, y_train,
    epochs=150,
    batch_size=64,
    verbose=1,
    validation_data=(X_val, y_val),
    shuffle=True,
    callbacks=[early_tr, ckpt_tr]
)

model_tr = load_model(MODELO_OUT_TR, custom_objects={"PositionalEncoding": PositionalEncoding})
models_trained["Transformer"] = model_tr
histories["Transformer"] = history_tr
print("✅ Best Transformer loaded:", MODELO_OUT_TR)


MAMBA

In [ ]:
early_mb, ckpt_mb, MODELO_OUT_MB = make_callbacks("Mamba")

@tf.keras.utils.register_keras_serializable()
class MambaBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, d_state, **kwargs):
        super().__init__(**kwargs)
        self.d_model = int(d_model)
        self.d_state = int(d_state)

    def build(self, input_shape):
        self.proj_in  = Dense(self.d_model * 2)
        self.conv1d   = Conv1D(self.d_model * 2, kernel_size=3, padding='causal')
        self.proj_out = Dense(self.d_model)
        self.norm     = LayerNormalization()
        in_dim = int(input_shape[-1])
        self.res_proj = Dense(self.d_model) if in_dim != self.d_model else None

    def call(self, x):
        residual = x
        x = self.proj_in(x)
        x = self.conv1d(x)

        x, gate = tf.split(x, 2, axis=-1)
        x = x * tf.nn.sigmoid(x)
        gate = tf.nn.sigmoid(gate)
        x = x * gate

        x = self.proj_out(x)

        if self.res_proj is not None:
            residual = self.res_proj(residual)

        return self.norm(x + residual)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"d_model": self.d_model, "d_state": self.d_state})
        return cfg

def build_mamba(input_shape, d_model=32, d_state=16, num_layers=2, dropout=0.1):
    inputs = Input(shape=input_shape)
    x = Dense(d_model)(inputs)
    for _ in range(num_layers):
        x = MambaBlock(d_model, d_state)(x)
        x = Dropout(dropout)(x)
    x = GlobalAveragePooling1D()(x)
    outputs = Dense(1, activation='sigmoid')(x)
    return Model(inputs, outputs, name="Mamba_SOC")

model_mb = build_mamba((X_train.shape[1], X_train.shape[2]), d_model=32, d_state=16, num_layers=2, dropout=0.1)
model_mb.compile(optimizer='adam', loss='mse')

history_mb = model_mb.fit(
    X_train, y_train,
    epochs=150,
    batch_size=64,
    verbose=1,
    validation_data=(X_val, y_val),
    shuffle=True,
    callbacks=[early_mb, ckpt_mb]
)

model_mb = load_model(MODELO_OUT_MB, custom_objects={"MambaBlock": MambaBlock})
models_trained["Mamba"] = model_mb
histories["Mamba"] = history_mb
print("✅ Best Mamba loaded:", MODELO_OUT_MB)


AVALIAÇÂO

In [ ]:
rows = []

for model_name, model_obj in models_trained.items():
    print(f"\n➡️ Avaliando: {model_name}")
    for ciclo in CICLOS:
        for t in TEMPS:
            arq = f"{DATA_DIR}/teste1_LSTM_{t}.xlsx"
            df_c = carregar_df(arq, ciclo)

            X_raw = x_scaler.transform(df_c[INPUT_COLS])
            y_raw = y_scaler.transform(df_c[[OUTPUT_COL]])

            X_seq, y_seq = criar_janelas_seq(X_raw, y_raw, TIME_STEP)
            if len(X_seq) == 0:
                continue

            t0 = time.time()
            y_pred_scaled = model_obj.predict(X_seq, verbose=0)
            t1 = time.time()

            infer_total_s = float(t1 - t0)
            infer_s_per_sample = infer_total_s / float(len(X_seq))
            infer_ms_per_sample = infer_s_per_sample * 1000.0

            y_pred_inv = y_scaler.inverse_transform(y_pred_scaled).ravel()
            y_true_inv = y_scaler.inverse_transform(y_seq).ravel()

            rmse, mae, maxe, r2 = metricas(y_true_inv, y_pred_inv)

            rows.append({
                "Model": model_name,
                "Cycle": ciclo,
                "Temp": t,
                "RMSE": rmse,
                "MAE": mae,
                "MaxError": maxe,
                "R2": r2,
                "Inference_ms_per_sample": infer_ms_per_sample,
                "N_samples": int(len(X_seq))
            })

df_results = pd.DataFrame(rows)

ts = time.strftime("%Y%m%d_%H%M%S")
out_csv = f"{OUT_DIR}/TEST2_modelos_crus_metrics_{ts}.csv"
df_results.to_csv(out_csv, index=False)

print("✅ Saved:", out_csv)
df_results



➡️ Avaliando: LSTM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler 

✅ Saved: relatorios/TEST2_modelos_crus_metrics_20260127_194001.csv


,Model,Cycle,Temp,RMSE,MAE,MaxError,R2,Inference_ms_per_sample,N_samples
0,LSTM,US06,10,0.018658,0.014152,0.072336,0.994529,0.298674,6550
1,LSTM,US06,20,0.014099,0.010518,0.055650,0.996882,0.210051,6558
2,LSTM,US06,30,0.011209,0.008262,0.045065,0.998030,0.398715,6549
3,LSTM,US06,40,0.010802,0.008356,0.044383,0.998157,0.216426,6552
4,LSTM,FUDS,10,0.015408,0.011247,0.082689,0.996009,0.231737,6799
5,LSTM,FUDS,20,0.012646,0.009539,0.051398,0.997303,0.226106,6799
6,LSTM,FUDS,30,0.011995,0.009051,0.053651,0.997563,0.210048,6791
7,LSTM,FUDS,40,0.011456,0.008381,0.055877,0.997808,0.232323,6798


In [ ]:
df_summary_model = (
    df_results.groupby("Model")[["RMSE","MAE","MaxError","R2","Inference_ms_per_sample"]]
    .mean().reset_index()
    .sort_values(["RMSE","MAE"], ascending=True)
)

df_summary_model_cycle = (
    df_results.groupby(["Model","Cycle"])[["RMSE","MAE","MaxError","R2","Inference_ms_per_sample"]]
    .mean().reset_index()
    .sort_values(["Cycle","RMSE","MAE"], ascending=[True, True, True])
)

print("=== Média por Modelo (geral) ===")
display(df_summary_model)

print("\n=== Média por Modelo e Ciclo ===")
display(df_summary_model_cycle)

out_xlsx = f"{OUT_DIR}/TEST2_modelos_crus_resumo_{ts}.xlsx"
with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as wr:
    df_results.to_excel(wr, sheet_name="results_by_temp", index=False)
    df_summary_model.to_excel(wr, sheet_name="summary_model", index=False)
    df_summary_model_cycle.to_excel(wr, sheet_name="summary_model_cycle", index=False)

print("✅ Excel saved:", out_xlsx)


=== Média por Modelo (geral) ===


,Model,RMSE,MAE,MaxError,R2,Inference_ms_per_sample
0,LSTM,0.013284,0.009938,0.057631,0.997035,0.25301



=== Média por Modelo e Ciclo ===


,Model,Cycle,RMSE,MAE,MaxError,R2,Inference_ms_per_sample
0,LSTM,FUDS,0.012876,0.009554,0.060904,0.997171,0.225053
1,LSTM,US06,0.013692,0.010322,0.054358,0.996900,0.280967


✅ Excel saved: relatorios/TEST2_modelos_crus_resumo_20260127_194001.xlsx
